# Phase 10: Business Intelligence & KPIs

Computes the headline KPI set used in the final report and the Streamlit app's Business Overview tab.

## Setup

In [1]:
import pandas as pd, numpy as np, json

df = pd.read_csv('../data/polokwane_sales_clean.csv', parse_dates=['date'])
sales = df[df['value_zar'] > 0].copy()
returns = df[df['value_zar'] < 0].copy()
out = '../outputs'

total_revenue = sales['value_zar'].sum()
total_return_value = -returns['value_zar'].sum()
return_rate = total_return_value / (total_revenue + total_return_value)

n_docs = sales['doc_number'].nunique()
avg_basket_value = sales.groupby('doc_number')['value_zar'].sum().mean()
avg_line_value = sales['value_zar'].mean()

# YoY growth (full calendar years only: 2024 vs 2023, comparable period 2025 vs 2024)
yearly_rev = sales.groupby(sales['date'].dt.year)['value_zar'].sum()
yoy_2024 = (yearly_rev[2024] / yearly_rev[2023] - 1) * 100
yoy_2025 = (yearly_rev[2025] / yearly_rev[2024] - 1) * 100

# Active customers per month (named accounts, i.e. excluding generic tills) -- retention proxy
generic_pattern = 'Cash Account|Transfers Customer|Pensioner Discount'
named_sales = sales[~sales['debtor_name'].str.contains(generic_pattern, case=False, regex=True)]
monthly_active = named_sales.groupby(named_sales['date'].dt.to_period('M'))['debtor_code'].nunique()

# Product concentration (from phase 8 outputs)
abc = pd.read_csv(f'{out}/product_abc_summary.csv', index_col=0)

kpis = {
    'total_revenue_zar': round(float(total_revenue), 2),
    'total_transactions': int(len(sales)),
    'unique_invoices': int(n_docs),
    'avg_basket_value_zar': round(float(avg_basket_value), 2),
    'avg_line_item_value_zar': round(float(avg_line_value), 2),
    'return_rate_pct': round(float(return_rate * 100), 3),
    'yoy_revenue_growth_2024_pct': round(float(yoy_2024), 1),
    'yoy_revenue_growth_2025_pct': round(float(yoy_2025), 1),
    'active_named_customers_avg_per_month': round(float(monthly_active.mean()), 1),
    'class_A_products_pct_of_revenue': float(abc.loc['A', 'pct_of_revenue']),
    'unique_products_sold': int(sales['product_code'].nunique()) if 'product_code' in sales else None,
    'unique_customer_accounts': int(sales['debtor_code'].nunique()),
}

with open(f'{out}/kpis.json', 'w') as f:
    json.dump(kpis, f, indent=2)

print(json.dumps(kpis, indent=2))

{
  "total_revenue_zar": 98282583.78,
  "total_transactions": 535920,
  "unique_invoices": 8513,
  "avg_basket_value_zar": 11545.0,
  "avg_line_item_value_zar": 183.39,
  "return_rate_pct": 0.226,
  "yoy_revenue_growth_2024_pct": 80.7,
  "yoy_revenue_growth_2025_pct": 6.8,
  "active_named_customers_avg_per_month": 17.5,
  "class_A_products_pct_of_revenue": 79.9,
  "unique_products_sold": 1616,
  "unique_customer_accounts": 101
}
